# poesia — LoRA fine-tuning on Colab's free T4

Companion to `docs/TRAINING_RUNBOOK.md` ("Online training" section) — read that first if anything here is unclear.

**What this notebook does:** clones the repo onto your Google Drive (so it survives a disconnect), installs the training deps, points MLflow at a SQLite file on Drive, and runs `scripts/train_poetry_lora.py` with checkpoint-resume enabled.

**Free-tier constraints to plan around:**
- GPU: a single NVIDIA T4 (Turing, 2018, 16 GB VRAM) — fine for 1.5B QLoRA, tight for the 3B model in 4-bit.
- Session cap: ~12 hours max, whether or not you're using it.
- Idle timeout: ~90 minutes of no interaction disconnects the runtime.
- No fixed guaranteed daily GPU quota on the free tier — availability varies.

None of this is something to fight around — it's why every cell below is designed so a disconnect costs you at most the time since the last checkpoint, not the whole run. See `save_steps` in the config `.yaml` files for how often that checkpoint happens.

**Testing without a real T4 GPU:** every cell up through "Install dependencies" and "Sanity-check the dataset" runs fine on Colab's free CPU runtime — only the actual `trainer.train()` cell needs the GPU runtime (Runtime → Change runtime type → T4 GPU). Switch to CPU runtime first to dry-run everything except that one cell.

## 1. Confirm the GPU runtime

In [1]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

Mon Aug 31 23:17:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Drive (persistent storage across disconnects)

Everything that needs to survive a disconnect — the repo clone, model checkpoints, the MLflow SQLite db — lives under `/content/drive/MyDrive/poesia`, not on the ephemeral Colab local disk.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/poesia"

MessageError: Error: credential propagation was unsuccessful

## 3. Clone the repo onto Drive (first run only)

You'll be prompted for a GitHub personal access token (repo scope). It's only
used for this `git clone`/`git pull` and is never written to disk or to this
notebook — enter it fresh each session via the hidden prompt below.

In [ ]:
import os
from getpass import getpass

GITHUB_REPO = "OomAngel/poesia"

if not os.path.exists(PROJECT_DIR):
    token = getpass("GitHub personal access token (repo scope): ")
    !git clone https://{token}@github.com/{GITHUB_REPO}.git {PROJECT_DIR}
    del token
else:
    print(f"{PROJECT_DIR} already exists on Drive — reusing it.")
    print("Run the next cell if you want to pull the latest commits first.")

%cd {PROJECT_DIR}

In [ ]:
# Optional: pull latest commits into an already-cloned Drive copy.
# Uncomment and run if PROJECT_DIR already existed above.

# token = getpass("GitHub personal access token (repo scope): ")
# !git remote set-url origin https://{token}@github.com/{GITHUB_REPO}.git
# !git pull
# !git remote set-url origin https://github.com/{GITHUB_REPO}.git  # scrub the token back out
# del token

## 4. Install dependencies

Colab already ships a CUDA-enabled `torch` — don't reinstall it. These are the
same versions confirmed working in the project's local conda env
(`environment.yml` doesn't pin these, they were installed separately — see
the note flagged alongside this notebook).

In [ ]:
!pip install -q -e .
!pip install -q "peft==0.20.0" "bitsandbytes==0.50.0" "accelerate==1.14.0" \
    "datasets==5.0.1" "mlflow==3.14.0" "transformers==5.14.1" "pyyaml==6.0.3"

## 5. Point MLflow at a SQLite file on Drive

The real tracking store is Postgres on the dev machine, which Colab can't
reach. `scripts/train_poetry_lora.py` reads `DATABASE_URL`; pointing it at a
SQLite file under `PROJECT_DIR` (on Drive) means the run history survives a
disconnect too, and can be merged into the real Postgres store afterward with
`mlflow-export-import` rather than lost.

In [ ]:
import os

os.environ["DATABASE_URL"] = f"sqlite:///{PROJECT_DIR}/mlruns/colab_mlflow.db"
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
print("MLflow tracking URI:", os.environ["DATABASE_URL"])

## 6. Pick a config and sanity-check the dataset (CPU-safe)

Start with `train_smoke.yaml` to confirm the whole pipeline runs end-to-end
on Colab before committing session hours to a bigger config like
`train_v2_fixed.yaml` or `train_repair.yaml`.

`train_repair.yaml` trains the repair role specifically (defect->fix line
repair, see `docs/GENERATION_QUALITY_PLAN.md` gap #9) on
`mlops/data/train_repair.jsonl` / `eval_repair.jsonl` — regenerate those
with `python scripts/format_repair_examples.py --split` if the source
repair data changes.


In [ ]:
import yaml

CONFIG_PATH = "mlops/configs/train_repair.yaml"  # swap to train_smoke.yaml/train_v2_fixed.yaml for other runs

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

print("model:", cfg["model"])
print("train_data:", cfg["train_data"])
print("epochs:", cfg.get("epochs"), "| batch_size:", cfg.get("batch_size"))
print("save_steps:", cfg.get("save_steps"), "(checkpoint frequency)")

with open(cfg["train_data"]) as f:
    n_records = sum(1 for _ in f)
print(f"train_data record count: {n_records}")


## 7. Train (GPU runtime required from here on)

`--resume-from-checkpoint` is always safe to pass, even on a fresh run — it's
a no-op if `output_dir` has no checkpoint yet, and picks up from the newest
one if it does. Re-run this exact cell after any disconnect.

In [ ]:
!python scripts/train_poetry_lora.py {CONFIG_PATH} --resume-from-checkpoint

## 8. After training

- The adapter is at `{output_dir}/final_adapter` under `PROJECT_DIR` on Drive.
- MLflow run history is in `PROJECT_DIR/mlruns/colab_mlflow.db` (SQLite).
- Back on the dev machine: copy the adapter directory over (or `dvc add` it
  from a Drive-synced path), and merge the SQLite run history into the real
  Postgres tracking store with `mlflow-export-import` rather than only
  copying the final adapter artifact.

In [ ]:
# Optional post-training sanity check: load the adapter and generate one line,
# so a corrupt/incomplete artifact is caught before you leave the session.

# import sys
# sys.path.insert(0, "src")
# from poesia.generation.llm_client import LoRAClient
#
# adapter_path = f"{cfg['output_dir']}/final_adapter"
# client = LoRAClient(adapter_path=adapter_path, base_model=cfg["model"])
# print(client.generate("Escribe un verso sobre la luna.", n=1))